# AF2CUE1 — validation-only Pareto post-hoc
Membaca laporan seed-42 yang sudah ada untuk analisis per kelas dan bootstrap komposisi 21 kelas. Tidak ada training, GPU, checkpoint inference, atau akses test. Keputusan frozen tetap `FAIL_KILL_GATE`; analisis ini hanya menilai apakah AF2CUE1 layak dipertahankan sebagai kandidat eksploratif.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, sys
from pathlib import Path
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'
BRANCH='codex/af2-radially-normalized-angular-density'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],cwd=WORK,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); os.chdir(REPO)


In [ ]:
from coffee_detector.drive_project import resolve_drive_project_root
BASE_REL='experiments/faruq-v3-af2-spds-v1/val_reports/AF2BASE_seed42_result.json'
SPDS_REL='experiments/faruq-v3-af2-spds-v1/val_reports/AF2SPDS_seed42_result.json'
CUE_REL='experiments/faruq-v3-af2-spds-refinement-v1/val_reports/AF2CUE1_seed42_result.json'
DECAY_REL='experiments/faruq-v3-af2-spds-refinement-v1/val_reports/AF2DECAY1_seed42_result.json'
PROJECT=resolve_drive_project_root(required_relative_paths=(BASE_REL,SPDS_REL,CUE_REL,DECAY_REL))
ORIGINAL=PROJECT/'experiments/faruq-v3-af2-spds-v1'
REFINEMENT=PROJECT/'experiments/faruq-v3-af2-spds-refinement-v1'
OUTPUT=REFINEMENT/'val_reports'/'af2cue1_seed42_posthoc.json'
print('PROJECT:',PROJECT); print('OUTPUT:',OUTPUT)


In [ ]:
from coffee_detector.analysis.af2_spds_refinement_posthoc import run_af2_spds_refinement_posthoc
result=run_af2_spds_refinement_posthoc(ORIGINAL,REFINEMENT,OUTPUT,iterations=10000,seed=20260829,print_json=False)
print('FORMAL DECISION :',result['formal_frozen_decision'])
print('EXPLORATORY     :',result['exploratory_research_status'])
print('CLASS SUMMARY   :',{k:v for k,v in result['class_summary'].items() if not isinstance(v,list)})
print('TRAINING:',result['training_executed'],'| TEST:',result['test_opened'])
print('SAVED:',OUTPUT)


In [ ]:
import pandas as pd
rows=pd.DataFrame(result['per_class'])
cols=['class_name','AF2BASE','AF2SPDS','AF2CUE1','cue1_minus_base','cue1_minus_spds']
print('GAIN TERBESAR AF2CUE1 vs AF2BASE')
display(rows.sort_values('cue1_minus_base',ascending=False)[cols].head(10).style.format({c:'{:+.2%}' for c in cols[1:]}))
print('PENURUNAN TERBESAR AF2CUE1 vs AF2SPDS')
display(rows.sort_values('cue1_minus_spds')[cols].head(10).style.format({c:'{:+.2%}' for c in cols[1:]}))


In [ ]:
bootstrap=result['paired_class_bootstrap']
for comparison,payload in bootstrap.items():
    print('\n'+comparison)
    display(pd.DataFrame(payload['metrics']).T.style.format('{:.4f}'))
print('CATATAN:',bootstrap['AF2CUE1_vs_AF2BASE']['independence_warning'])
print('Kirim exploratory status, class summary, dua tabel kelas, dan bootstrap. Jangan training atau membuka test.')
